# Лабораторная работа 8

Численное решение волнового уравнения

Вариант 21:

- `U_tt = D^2 U_xx + f(t,x)`
- `D = 1`, `f(t,x)=0`
- `U(x,0)=x^2`
- `U_t(x,0)=1`
- `U(0,t)=0`, `U(1,t)=1`
- область: `x ∈ [0,1]`, `t ∈ [0,10]`


In [1]:
import contextlib
import io
import json
from pathlib import Path

import matplotlib
import numpy as np
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt
from matplotlib import pyplot as plt

matplotlib.use("Agg")

BASE_DIR = Path(__file__).resolve().parent
NOTEBOOK_PATH = BASE_DIR / "8.ipynb"
DOCX_PATH = BASE_DIR / "8_Туманов.docx"
PLOTS_DIR = BASE_DIR / "plots"

X0 = 0.0
X1 = 1.0
T0 = 0.0
T1 = 10.0
D = 1.0
EPS = 0.01
H = 0.1
TAU = 0.1
NX = int(round((X1 - X0) / H))
NT = int(round((T1 - T0) / TAU))
R = (D * TAU / H) ** 2
EXACT_TERMS = 2000

X = np.linspace(X0, X1, NX + 1)
T = np.linspace(T0, T1, NT + 1)


def phi(x):
    return x * x


def psi(x):
    return np.ones_like(x, dtype=float)


def left_boundary(t):
    return 0.0


def right_boundary(t):
    return 1.0


def source(t, x):
    return 0.0


def exact_value(x, t, terms=EXACT_TERMS):
    n = np.arange(1, terms + 1, dtype=float)
    sin_nx = np.sin(np.pi * n * x)
    a_n = 4.0 * ((-1.0) ** n - 1.0) / (np.pi**3 * n**3)
    b_n = 2.0 * (1.0 - (-1.0) ** n) / (np.pi**2 * n**2)
    return x + np.sum((a_n * np.cos(np.pi * n * t) + b_n * np.sin(np.pi * n * t)) * sin_nx)


def build_exact_grid():
    u = np.zeros((NT + 1, NX + 1))
    for j, tt in enumerate(T):
        for i, xx in enumerate(X):
            u[j, i] = exact_value(xx, tt)
    return u


def init_grid():
    u = np.zeros((NT + 1, NX + 1))
    u[0, :] = phi(X)
    for j, tt in enumerate(T):
        u[j, 0] = left_boundary(tt)
        u[j, NX] = right_boundary(tt)

    # Formula of Taylor expansion with second-order accuracy.
    phi_second = 2.0
    for i in range(1, NX):
        u[1, i] = phi(X[i]) + TAU * 1.0 + 0.5 * TAU**2 * (D**2 * phi_second + source(0.0, X[i]))
    return u


def solve_explicit():
    u = init_grid()
    for j in range(1, NT):
        u[j + 1, 0] = left_boundary(T[j + 1])
        u[j + 1, NX] = right_boundary(T[j + 1])
        for i in range(1, NX):
            u[j + 1, i] = (
                2.0 * u[j, i]
                - u[j - 1, i]
                + R * (u[j, i + 1] - 2.0 * u[j, i] + u[j, i - 1])
                + TAU**2 * source(T[j], X[i])
            )
    return u


def thomas(a, b, c, d):
    n = len(d)
    alpha = np.zeros(n)
    beta = np.zeros(n)
    result = np.zeros(n)

    alpha[0] = -c[0] / b[0]
    beta[0] = d[0] / b[0]
    for i in range(1, n):
        denominator = b[i] + a[i] * alpha[i - 1]
        alpha[i] = 0.0 if i == n - 1 else -c[i] / denominator
        beta[i] = (d[i] - a[i] * beta[i - 1]) / denominator

    result[-1] = beta[-1]
    for i in range(n - 2, -1, -1):
        result[i] = alpha[i] * result[i + 1] + beta[i]
    return result


def solve_implicit_upper():
    u = init_grid()
    size = NX - 1
    for j in range(1, NT):
        a = np.full(size, -R)
        b = np.full(size, 1.0 + 2.0 * R)
        c = np.full(size, -R)
        d = np.zeros(size)

        for i in range(1, NX):
            d[i - 1] = 2.0 * u[j, i] - u[j - 1, i] + TAU**2 * source(T[j + 1], X[i])

        d[0] -= a[0] * left_boundary(T[j + 1])
        a[0] = 0.0
        d[-1] -= c[-1] * right_boundary(T[j + 1])
        c[-1] = 0.0

        inner = thomas(a, b, c, d)
        u[j + 1, 0] = left_boundary(T[j + 1])
        u[j + 1, NX] = right_boundary(T[j + 1])
        u[j + 1, 1:NX] = inner
    return u


def solve_implicit_rectangle():
    u = init_grid()
    size = NX - 1
    for j in range(1, NT):
        a = np.full(size, -0.5 * R)
        b = np.full(size, 1.0 + R)
        c = np.full(size, -0.5 * R)
        d = np.zeros(size)

        for i in range(1, NX):
            d[i - 1] = (
                2.0 * u[j, i]
                - (1.0 + R) * u[j - 1, i]
                + 0.5 * R * (u[j - 1, i - 1] + u[j - 1, i + 1])
                + TAU**2 * source(T[j], X[i])
            )

        d[0] -= a[0] * left_boundary(T[j + 1])
        a[0] = 0.0
        d[-1] -= c[-1] * right_boundary(T[j + 1])
        c[-1] = 0.0

        inner = thomas(a, b, c, d)
        u[j + 1, 0] = left_boundary(T[j + 1])
        u[j + 1, NX] = right_boundary(T[j + 1])
        u[j + 1, 1:NX] = inner
    return u


def max_abs_error(numeric, exact):
    return float(np.max(np.abs(numeric - exact)))


def sample_at(u, x_value, t_value):
    i = int(round((x_value - X0) / H))
    j = int(round((t_value - T0) / TAU))
    return float(u[j, i])


def plot_heatmap(u, title, filename):
    PLOTS_DIR.mkdir(exist_ok=True)
    x_mesh, t_mesh = np.meshgrid(X, T)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    im = ax.pcolormesh(x_mesh, t_mesh, u, shading="auto", cmap="magma")
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="U")
    fig.tight_layout()
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def plot_surface(u, title, filename):
    PLOTS_DIR.mkdir(exist_ok=True)
    x_mesh, t_mesh = np.meshgrid(X, T)
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(x_mesh, t_mesh, u, cmap="magma", edgecolor="none")
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    ax.set_zlabel("U")
    ax.set_title(title)
    ax.view_init(elev=25, azim=-60)
    fig.colorbar(surf, shrink=0.55, aspect=12)
    fig.tight_layout()
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def plot_profile_comparison(exact, explicit, implicit_upper, implicit_rect, t_value, filename):
    PLOTS_DIR.mkdir(exist_ok=True)
    j = int(round(t_value / TAU))
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(X, exact[j], "k--", linewidth=2, label="Точное решение")
    ax.plot(X, explicit[j], linewidth=2, label="Явная схема")
    ax.plot(X, implicit_upper[j], linewidth=2, label="Неявная схема")
    ax.plot(X, implicit_rect[j], linewidth=2, label="Прямоугольная схема")
    ax.set_xlabel("x")
    ax.set_ylabel("U")
    ax.set_title(f"Сравнение профилей при t = {t_value}")
    ax.grid(True, alpha=0.35)
    ax.legend()
    fig.tight_layout()
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def create_all_plots(exact, explicit, implicit_upper, implicit_rect):
    return [
        plot_heatmap(exact, "Точное решение", "heat_exact.png"),
        plot_heatmap(explicit, "Явная схема", "heat_explicit.png"),
        plot_heatmap(implicit_upper, "Неявная схема", "heat_implicit_upper.png"),
        plot_heatmap(implicit_rect, "Неявная прямоугольная схема", "heat_implicit_rectangle.png"),
        plot_surface(explicit, "Явная схема", "surface_explicit.png"),
        plot_surface(implicit_upper, "Неявная схема", "surface_implicit_upper.png"),
        plot_surface(implicit_rect, "Неявная прямоугольная схема", "surface_implicit_rectangle.png"),
        plot_profile_comparison(exact, explicit, implicit_upper, implicit_rect, 1.5, "profile_t15.png"),
    ]


def compute_results():
    exact = build_exact_grid()
    explicit = solve_explicit()
    implicit_upper = solve_implicit_upper()
    implicit_rect = solve_implicit_rectangle()
    errors = {
        "Явная": max_abs_error(explicit, exact),
        "Неявная": max_abs_error(implicit_upper, exact),
        "Неявная прямоугольная": max_abs_error(implicit_rect, exact),
    }
    times = [0.5, 1.5, 2.5, 5.5, 8.5]
    samples = []
    for tt in times:
        samples.append(
            {
                "t": tt,
                "exact": sample_at(exact, 0.5, tt),
                "explicit": sample_at(explicit, 0.5, tt),
                "implicit": sample_at(implicit_upper, 0.5, tt),
                "rectangle": sample_at(implicit_rect, 0.5, tt),
            }
        )
    return exact, explicit, implicit_upper, implicit_rect, errors, samples



In [2]:
exact, explicit, implicit_upper, implicit_rect, errors, samples = compute_results()
plots = create_all_plots(exact, explicit, implicit_upper, implicit_rect)

print(f'Сетка: h={H}, tau={TAU}, NX={NX}, NT={NT}, r={R:.2f}')
print('Максимальные абсолютные погрешности:')
for name, value in errors.items():
    print(f'{name}: {value:.6f}')
print()
print('Сравнение решений при x = 0.5:')
print(f"{'t':>6} {'exact':>10} {'explicit':>10} {'implicit':>10} {'rectangle':>12}")
for row in samples:
    print(f"{row['t']:6.1f} {row['exact']:10.3f} {row['explicit']:10.3f} {row['implicit']:10.3f} {row['rectangle']:12.3f}")
print()
print('Графики:')
for plot in plots:
    print(plot)


Сетка: h=0.1, tau=0.1, NX=10, NT=100, r=1.00
Максимальные абсолютные погрешности:
Явная: 0.000101
Неявная: 0.500091
Неявная прямоугольная: 0.390168

Сравнение решений при x = 0.5:
     t      exact   explicit   implicit    rectangle
   0.5      1.000      1.000      0.815        0.926
   1.5      0.000      0.000      0.323        0.171
   2.5      1.000      1.000      0.600        0.860
   5.5      0.000      0.000      0.485        0.272
   8.5      1.000      1.000      0.501        0.740

Графики:
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab8\plots\heat_exact.png
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab8\plots\heat_explicit.png
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab8\plots\heat_implicit_upper.png
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab8\plots\heat_implicit_rectangle.png
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab8\plots\surface_explicit.png
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab8\plots\surface_implic